# 01 — Baseline: YOLO Single-Stage Digit Detection

Direct digit detection using YOLOv11. Each digit is a separate bounding box (classes 0–9).
Sorted left-to-right → reconstructed meter reading.

In [ ]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path("../..").resolve()
if not (ROOT / "models").exists():
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/UrranQx/WaterMeterCV.git"], check=True)
    ROOT = Path("WaterMeterCV").resolve()

sys.path.insert(0, str(ROOT))

import yaml
import json
import time
import csv
import torch
import cv2
import numpy as np
from datetime import datetime
from ultralytics import YOLO

from models.data.unified_loader import load_utility_meter_dataset, load_water_meter_dataset
from models.metrics.evaluation import full_string_accuracy, per_digit_accuracy, character_error_rate
from models.utils.visualization import draw_digit_bboxes

print(f"ROOT: {ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL SIZE — change this to switch between nano/small/medium:
#   "yolo11n" — nano  (~2.6M params, ~15 ms/img)  ← start here
#   "yolo11s" — small (~9.6M params, ~45 ms/img)  ← better accuracy
#   "yolo11m" — medium (~20M params, ~120 ms/img) ← max quality, batch↓
# ═══════════════════════════════════════════════════════════════════
MODEL_SIZE = "yolo11n"

# Paths
DATASET_PATH = ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11"
WM_PATH = ROOT / "WaterMetricsDATA/waterMeterDataset/WaterMeters"
DATA_YAML = DATASET_PATH / "data.yaml"
WEIGHTS_DIR = ROOT / "models/weights/baseline_yolo"
RESULTS_DIR = ROOT / "results"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
EPOCHS = 50
BATCH_SIZE = 16  # reduce to 8–12 for yolo11m if OOM
IMG_SIZE = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATIENCE = 10

RUN_NAME = f"{MODEL_SIZE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Model: {MODEL_SIZE}")
print(f"Device: {DEVICE}")
print(f"Run name: {RUN_NAME}")

In [ ]:
# Load original data.yaml
with open(DATA_YAML) as f:
    data_config = yaml.safe_load(f)

# Fix paths: original uses ../train/images (relative to yaml),
# which breaks depending on CWD. Set absolute path to be safe.
data_config['path'] = str(DATASET_PATH)
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'

FIXED_YAML = WEIGHTS_DIR / "data.yaml"
with open(FIXED_YAML, 'w') as f:
    yaml.dump(data_config, f)

print(f"Classes ({data_config['nc']}): {data_config['names']}")
print(f"Fixed data.yaml saved to {FIXED_YAML}\n")

# Count images per split
for split_name, split_dir in [("train", "train"), ("valid", "valid"), ("test", "test")]:
    img_dir = DATASET_PATH / split_dir / "images"
    count = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    print(f"  {split_name}: {count} images")